In [1]:
!pip install scikit-surprise -q

import pandas as pd
import pickle
from collections import defaultdict

# Adjust this to wherever you've uploaded your Week 1 interim files as a Kaggle Dataset
INTERIM_PATH = "/kaggle/input/datasets/anurajgogoi/cineiq-interim-files/interim"

import os
print(os.listdir(INTERIM_PATH))

['movie_to_rt_mapping.parquet', 'ratings_val_warm.parquet', 'ratings_val.parquet', 'svd_model_dev.pkl', 'week1_config.json', 'ratings_test.parquet', 'rt_reviews_matched.parquet', 'ratings_train.parquet', 'imdb_reviews_clean.parquet', 'movies_master.parquet', 'warm_users.parquet', 'ratings_test_warm.parquet']


In [2]:
from surprise import Dataset, Reader

warm_users = set(pd.read_parquet(f"{INTERIM_PATH}/warm_users.parquet")["userId"])
print("warm users:", len(warm_users))

ratings_train = pd.read_parquet(f"{INTERIM_PATH}/ratings_train.parquet")
print("ratings_train before filter:", ratings_train.shape)

ratings_train = ratings_train[ratings_train["userId"].isin(warm_users)]
print("ratings_train after warm-user filter:", ratings_train.shape)

df_for_surprise = ratings_train[["userId", "movieId", "rating"]]
reader = Reader(rating_scale=(0.5, 5.0))
dataset = Dataset.load_from_df(df_for_surprise, reader)

trainset = dataset.build_full_trainset()
print("\nSurprise trainset n_users:", trainset.n_users)
print("Surprise trainset n_items:", trainset.n_items)
print("Surprise trainset n_ratings:", trainset.n_ratings)

warm users: 126591
ratings_train before filter: (20798765, 6)
ratings_train after warm-user filter: (20462986, 6)

Surprise trainset n_users: 126591
Surprise trainset n_items: 37334
Surprise trainset n_ratings: 20462986


In [3]:
# Train SVD on the full warm-user trainset
from surprise import SVD
import time

start = time.time()
model = SVD(n_factors=100, n_epochs=20, random_state=42)
model.fit(trainset)
elapsed = time.time() - start

print(f"training completed in {elapsed:.1f} seconds")

sample_row = ratings_train.iloc[0]
uid, iid, true_r = sample_row["userId"], sample_row["movieId"], sample_row["rating"]
pred = model.predict(uid, iid, r_ui=true_r)
print(f"\nsample prediction for userId={uid}, movieId={iid}:")
print(f"  true rating: {true_r}")
print(f"  predicted:   {pred.est:.3f}")
print(f"  {pred}")

training completed in 195.7 seconds

sample prediction for userId=1, movieId=296:
  true rating: 5.0
  predicted:   4.466
  user: 1          item: 296        r_ui = 5.00   est = 4.47   {'was_impossible': False}


In [4]:
# CELL 4 — Evaluate on the full warm-user validation set
from surprise import accuracy

val_warm = pd.read_parquet(f"{INTERIM_PATH}/ratings_val_warm.parquet")
print("val_warm shape:", val_warm.shape)

testset = list(zip(val_warm["userId"], val_warm["movieId"], val_warm["rating"]))

predictions = model.test(testset)
rmse = accuracy.rmse(predictions, verbose=True)

# Precision@10
def precision_at_k(predictions, k=10, threshold=3.5):
    user_est_true = defaultdict(list)
    for uid, iid, true_r, est, _ in predictions:
        user_est_true[uid].append((est, true_r))

    precisions = {}
    for uid, user_ratings in user_est_true.items():
        user_ratings.sort(key=lambda x: x[0], reverse=True)
        top_k = user_ratings[:k]
        n_relevant = sum((true_r >= threshold) for (_, true_r) in top_k)
        precisions[uid] = n_relevant / len(top_k) if top_k else 0

    return sum(precisions.values()) / len(precisions)

prec_at_10 = precision_at_k(predictions, k=10, threshold=3.5)
print(f"\nPrecision@10 (threshold=3.5): {prec_at_10:.4f}")
print(f"evaluated on {len(set(val_warm['userId']))} warm users, {len(val_warm)} ratings")

val_warm shape: (412348, 6)
RMSE: 0.8130

Precision@10 (threshold=3.5): 0.8254
evaluated on 5454 warm users, 412348 ratings


In [5]:
import pickle
import json

with open("/kaggle/working/svd_model.pkl", "wb") as f:
    pickle.dump(model, f)

results = {
    "model": "SVD",
    "n_factors": 100,
    "n_epochs": 20,
    "train_users": trainset.n_users,
    "train_items": trainset.n_items,
    "train_ratings": trainset.n_ratings,
    "val_ratings_evaluated": len(val_warm),
    "val_users_evaluated": len(set(val_warm["userId"])),
    "rmse": round(rmse, 4),
    "precision_at_10": round(prec_at_10, 4),
    "training_time_seconds": round(elapsed, 1),
}

with open("/kaggle/working/svd_results.json", "w") as f:
    json.dump(results, f, indent=2)

print(json.dumps(results, indent=2))
print("\nsaved to /kaggle/working/: svd_model.pkl, svd_results.json")

{
  "model": "SVD",
  "n_factors": 100,
  "n_epochs": 20,
  "train_users": 126591,
  "train_items": 37334,
  "train_ratings": 20462986,
  "val_ratings_evaluated": 412348,
  "val_users_evaluated": 5454,
  "rmse": 0.813,
  "precision_at_10": 0.8254,
  "training_time_seconds": 195.7
}

saved to /kaggle/working/: svd_model.pkl, svd_results.json
